In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Japan Earthquake Catalog — Pre-Analysis (JMA via ISC, 1985-2025)

Covers: catalog EDA, Gutenberg-Richter law, Omori-Utsu law.
Run as a script  : python JAPAN_preanalysis.py
Convert to notebook: python convert_to_notebook.py JAPAN_preanalysis.py notebooks/JAPAN_preanalysis.ipynb
"""

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.seismology import fit_gr_law, fit_omori
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR = Path("data/Japan")
CUT_YEAR = 1985
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "japan")

## Data Loading and Basic Statistics

In [ ]:
print("Loading Japan earthquake catalog (JMA via ISC)...")
df = pd.read_csv(DATA_DIR / "japan_earthquakes_jma_1985_2025_m1_5.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df["year"] = df["time"].dt.year
df_net = (
    df[df["year"] >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['year'].min()}–{df_net['year'].max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Columns: {list(df_net.columns)}")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)}")

## Temporal Distribution

Annual event counts reveal catalog completeness and major seismic episodes.
The 1985 cut year is set to avoid the sharp improvement in JMA detection
sensitivity in earlier decades; year-on-year rate stability is the key
quality indicator.

Japan is among the world's most seismically active regions. Dominant spikes
correspond to major sequences: **1995 Kobe** (M6.9), **2004 Chuetsu** (M6.8),
**2011 Tohoku** (M9.0, by far the largest — 2011 will be an extreme outlier),
and **2016 Kumamoto** (M7.0). The Tohoku sequence alone contributed
tens of thousands of aftershocks, making 2011 incomparable to other years
and important to account for in statistical analyses.

In [ ]:
year_counts = df_net["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(18, 5))
sns.barplot(x=year_counts.index, y=year_counts.values, color="steelblue", ax=ax)
ax.set_title("Number of Seismic Events (M ≥ 1.5) in Japan per Year", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Earthquakes")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_event_counts_japan")
plt.show()

fig, ax = plt.subplots(figsize=(18, 5))
sns.boxenplot(data=df_net, x="year", y="magnitude", palette="viridis", ax=ax)
ax.set_title("Magnitude Distribution per Year", fontsize=16)
ax.set_xlabel("Year")
ax.set_ylabel("Magnitude")
plt.xticks(rotation=45)
plt.tight_layout()
savefig("temporal_magnitude_dist_japan")
plt.show()

## Magnitude and Depth Analysis

The magnitude histogram should follow the Gutenberg-Richter exponential decay
above the completeness threshold $M_c$; the rapid drop below $M_c$ indicates
under-reporting. For JMA, $M_c \approx 1.0$ onshore and higher offshore.

Depth histograms for Japan reflect its complex tectonic setting:
a **shallow crustal peak** (0–30 km) from interplate and intraplate
earthquakes; a **mid-depth cluster** (30–100 km) from the subducting Pacific
and Philippine Sea slabs; and a **deep cluster** (> 300 km) unique to Japan's
subduction geometry. This three-layer structure should produce multiple
topologically distinct node populations in the Abe-Suzuki network.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df_net["magnitude"].dropna(), bins=np.arange(1.45, 9.55, 0.1),
             edgecolor="black", color="skyblue")
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Magnitude Distribution")

axes[1].hist(df_net["depth_km"].dropna(), bins=100, edgecolor="black", color="salmon")
axes[1].set_xlabel("Depth (km)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Depth Distribution")
plt.tight_layout()
savefig("magnitude_depth_distributions_japan")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df_net["magnitude"], df_net["depth_km"], alpha=0.04, s=2, color="crimson")
axes[0].invert_yaxis()
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Depth (km)")
axes[0].set_title("Magnitude vs Depth")

hb = axes[1].hexbin(df_net["magnitude"], df_net["depth_km"],
                    gridsize=50, cmap="inferno", mincnt=1, bins="log")
plt.colorbar(hb, ax=axes[1], label="log₁₀(count)")
axes[1].invert_yaxis()
axes[1].set_xlabel("Magnitude")
axes[1].set_ylabel("Depth (km)")
axes[1].set_title("Magnitude vs Depth (Density)")
plt.tight_layout()
savefig("magnitude_vs_depth_japan")
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
df_net["mag_bin"] = pd.cut(df_net["magnitude"], bins=[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 9.5])
sns.boxplot(data=df_net, x="mag_bin", y="depth_km", palette="coolwarm",
            showfliers=False, ax=ax)
ax.invert_yaxis()
ax.set_title("Depth Distribution by Magnitude Bin", fontsize=16)
ax.set_xlabel("Magnitude Range")
ax.set_ylabel("Depth (km)")
plt.tight_layout()
savefig("depth_by_magnitude_bin_japan")
plt.show()

## Seismicity Map

Interactive spatial overview of significant events (M ≥ 5) on a Japan basemap.
The $M \geq 5$ threshold retains the most damaging and best-located events while
keeping the map legible.

Expected pattern for Japan: dense offshore activity along the **Japan Trench**
(Pacific plate subduction, NE Japan), **Sagami Trough** (Philippine Sea plate,
Kanto region), and **Ryukyu Trench** (SW Japan). The 2011 Tohoku megathrust
zone (off Miyagi-Iwate coast) should dominate the scatter. Onshore activity
concentrates along the **Median Tectonic Line** and inland fault systems.

In [ ]:
mag_cut = 5
df_map = df_net[df_net["magnitude"] >= mag_cut].copy()
print(f"Mapping {len(df_map)} significant earthquakes (M ≥ {mag_cut})...")

_JP_BOUNDS = dict(west=122, east=154, south=24, north=46.5)

fig = px.scatter_map(
    df_map,
    lat="latitude", lon="longitude",
    color="magnitude", size="magnitude",
    color_continuous_scale="matter",
    map_style="carto-positron",
    hover_name="time",
    hover_data={"latitude": ":.3f", "longitude": ":.3f",
                "depth_km": True, "magnitude": True},
    title=f"Interactive Seismic Map: Japan (M ≥ {mag_cut})",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, height=700,
                  map=dict(center={"lat": 36.0, "lon": 138.0}, zoom=3.5,
                           bounds=_JP_BOUNDS))
save_plotly(fig, "seismicity_map_japan")
fig.show()

## Gutenberg-Richter Law

The Gutenberg-Richter relation states $\log_{10} N(\geq M) = a - bM$, where
$N(\geq M)$ is the cumulative count of earthquakes with magnitude $\geq M$,
$a$ (seismicity productivity) and $b \approx 1$ globally (Gutenberg & Richter 1944).

**Sensitivity test**: we vary the upper truncation $M_{\max}$ across
[4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0] to assess fit stability.
Japan's high seismicity rate should yield an extremely high $a$-value relative
to Italy or the US. The $b$-value is expected to be close to 1.0 — consistent
with the global average — though subduction-zone catalogs sometimes show
$b \approx 0.8$–$0.9$ due to stress concentration on locked megathrust patches.

In [ ]:
max_mags   = [4, 4.5, 5, 5.5, 6, 6.5, 7]
plot_flags = [True, False, False, True, False, False, True]

gr_results = [
    fit_gr_law(df_net, max_mag=m, plot=p)
    for m, p in zip(max_mags, plot_flags)
]
df_gr = pd.DataFrame(gr_results).sort_values("max_mag")
display(df_gr)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df_gr["max_mag"], df_gr["a_value"], "o-", color="tab:blue", label="a-value")
ax1.set_xlabel("Max Magnitude Used for Fit")
ax1.set_ylabel("a-value", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(df_gr["max_mag"], df_gr["b_value"], "s-", color="tab:red", label="b-value")
ax2.set_ylabel("b-value", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title("Gutenberg-Richter Parameters vs Max Magnitude")
ax1.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
savefig("gr_parameters_sensitivity_japan")
plt.show()

## Omori Law — Kobe 1995

The Omori-Utsu law describes aftershock rate decay: $n(t) = K / (c + t)^p$,
where $t$ is time since the mainshock, $p \approx 1$ globally (Utsu et al. 1995),
$K$ scales productivity, and $c$ prevents the singularity at $t = 0$.

The **1995 Kobe earthquake** (M6.9, 17 Jan 1995 JST, 34.583°N 135.039°E)
struck along the Nojima fault (Awaji Island) and the Rokko fault system,
causing 6,434 fatalities. It is the canonical benchmark sequence for Japanese
intraplate seismicity and Omori-law calibration (Utsu 1961). Expected:
$p \approx 1.0$–$1.1$ (crustal, warm, well-documented completeness).
The relatively short aftershock zone (≈ 50 km) makes the sequence
geographically compact and easy to isolate.

In [ ]:
print("Fitting Omori law for Kobe 1995 sequence...")
res_kobe = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("1995-01-16 20:46:52", utc=True),
    days=60,
    lat_range=(33.0, 36.5),
    lon_range=(133.0, 138.0),
    event_name="Kobe 1995",
    mag_cut=1.5,
)

## Omori Law — Tohoku 2011

The **2011 Tōhoku earthquake** (M9.0, 11 Mar 2011 UTC, 38.322°N 142.369°E)
is the largest instrumentally recorded earthquake in Japan and the fourth
largest globally. It ruptured ≈ 500 km of the Japan Trench megathrust,
triggering a devastating tsunami. The aftershock sequence is the most
extensive in the JMA catalog — tens of thousands of events in the first year.

Fitting $n(t) = K / (c + t)^p$ on the 90-day window tests Omori decay
at megathrust scale, where $p$ and $c$ may differ from crustal sequences
due to the larger rupture area, deep slip patches, and fluid interactions.
Comparing Kobe and Tohoku $p$-values quantifies the tectonic-regime effect
on aftershock decay — a key input to seismic hazard assessment.

In [ ]:
print("Fitting Omori law for Tohoku 2011 sequence...")
res_tohoku = fit_omori(
    df_net,
    mainshock_time=pd.to_datetime("2011-03-11 05:46:24", utc=True),
    days=90,
    lat_range=(35.0, 42.0),
    lon_range=(138.0, 146.0),
    event_name="Tohoku 2011",
    mag_cut=1.5,
)

df_omori = pd.DataFrame([
    {"event": "Kobe 1995",   **res_kobe},
    {"event": "Tohoku 2011", **res_tohoku},
])
print("\nOmori fit comparison:")
display(df_omori)